# Step 4 — Model Training (5-Fold CV)

Train one of the five models using 5-fold cross-validation on the augmented train-val set.
Set `MODEL_NAME` below to choose which model to train.

| Model | Magnification | Architecture     |
|-------|---------------|------------------|
| M1    | 40X           | ResNet50         |
| M2    | 100X          | EfficientNet-B3  |
| M3    | 200X          | EfficientNet-B4  |
| M4    | 400X          | Swin Transformer |
| M5    | All (40–400X) | ResNet50         |

### Configuration

Set `MODEL_NAME` to one of `M1`–`M5`. All other hyperparameters have sensible defaults.
Flip `USE_COLAB = True` for Google Drive paths.

In [ ]:
import sys
from pathlib import Path

# ---------- toggle ----------
USE_COLAB = False
MODEL_NAME = "M1"

if USE_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Pigeon_VS_AI")
else:
    PROJECT_ROOT = Path(".")

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

# ---------- model configs ----------
MODEL_CONFIGS = {
    "M1": {"arch": "resnet50",         "magnification": 40,   "input_size": 224, "blur_compress": False},
    "M2": {"arch": "efficientnet_b3",  "magnification": 100,  "input_size": 224, "blur_compress": False},
    "M3": {"arch": "efficientnet_b4",  "magnification": 200,  "input_size": 224, "blur_compress": False},
    "M4": {"arch": "swin_t",           "magnification": 400,  "input_size": 224, "blur_compress": False},
    "M5": {"arch": "resnet50",         "magnification": None, "input_size": 224, "blur_compress": True},
}

# ---------- hyperparameters ----------
BATCH_SIZE = 32
NUM_EPOCHS = 30
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5
NUM_WORKERS = 2
N_FOLDS = 5

cfg = MODEL_CONFIGS[MODEL_NAME]
print(f"Training {MODEL_NAME}: arch={cfg['arch']}, mag={cfg['magnification']}, blur_compress={cfg['blur_compress']}")

### Load manifest and build model

Read the training manifest from step 3, then construct the selected architecture with pretrained ImageNet weights.
The final classification head is replaced with a single-output sigmoid for binary classification (benign vs malignant).

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

manifest = pd.read_csv(DATA_DIR / "train_manifest.csv")
print(f"Manifest: {len(manifest)} rows")


def build_model(arch: str) -> nn.Module:
    if arch == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, 1)
    elif arch == "efficientnet_b3":
        model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)
    elif arch == "efficientnet_b4":
        model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 1)
    elif arch == "swin_t":
        model = models.swin_t(weights=models.Swin_T_Weights.DEFAULT)
        model.head = nn.Linear(model.head.in_features, 1)
    else:
        raise ValueError(f"Unknown architecture: {arch}")
    return model


test_model = build_model(cfg["arch"])
total_params = sum(p.numel() for p in test_model.parameters())
print(f"Architecture: {cfg['arch']}, params: {total_params:,}")
del test_model

### Training and validation functions

One epoch of training with BCEWithLogitsLoss, and a validation pass that returns loss and accuracy.
The model outputs raw logits (single neuron); sigmoid is applied by the loss function during training
and manually during validation for predictions.

In [ ]:
from torch.utils.data import DataLoader


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        preds = (logits > 0).long()
        correct += (preds == labels.long()).sum().item()
        total += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().to(device)
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        preds = (logits > 0).long()
        correct += (preds == labels.long()).sum().item()
        total += len(labels)
    return total_loss / total, correct / total

### 5-fold training loop

For each fold:
1. Build a fresh model with pretrained weights
2. Split manifest into train (4 folds) and val (1 fold)
3. Train up to `NUM_EPOCHS` with early stopping on validation loss
4. Save the best checkpoint per fold to `models/{MODEL_NAME}_fold{f}.pt`

All images from a slide stay in the same partition — no leakage.

In [ ]:
from breakhis_dataset import BreakHisDataset

fold_results = []

for fold in range(N_FOLDS):
    print(f"\n{'='*60}")
    print(f"Fold {fold}/{N_FOLDS - 1}")
    print(f"{'='*60}")

    train_df = manifest[manifest["fold"] != fold]
    val_df = manifest[manifest["fold"] == fold]

    train_ds = BreakHisDataset(
        train_df,
        magnification=cfg["magnification"],
        input_size=cfg["input_size"],
        blur_and_compress=cfg["blur_compress"],
    )
    val_ds = BreakHisDataset(
        val_df,
        magnification=cfg["magnification"],
        input_size=cfg["input_size"],
        blur_and_compress=False,
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    print(f"Train: {len(train_ds)} images, Val: {len(val_ds)} images")

    model = build_model(cfg["arch"]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    criterion = nn.BCEWithLogitsLoss()

    best_val_loss = float("inf")
    patience_counter = 0
    save_path = MODEL_DIR / f"{MODEL_NAME}_fold{fold}.pt"

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = validate(model, val_loader, criterion)
        scheduler.step()

        print(f"  Epoch {epoch:2d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    final_val_loss, final_val_acc = validate(model, val_loader, criterion)
    fold_results.append({"fold": fold, "val_loss": final_val_loss, "val_acc": final_val_acc})
    print(f"  Best — val_loss={final_val_loss:.4f} val_acc={final_val_acc:.4f} → saved {save_path.name}")

    del model, optimizer, scheduler
    torch.cuda.empty_cache()

### Cross-validation summary

Aggregate validation accuracy across all 5 folds. The mean and std give a robust estimate of model performance before touching the test set.

In [ ]:
results_df = pd.DataFrame(fold_results)
print(f"\n{MODEL_NAME} — 5-Fold CV Results")
print(f"{'='*40}")
print(results_df.to_string(index=False))
print(f"\nMean val_acc: {results_df['val_acc'].mean():.4f} ± {results_df['val_acc'].std():.4f}")
print(f"Mean val_loss: {results_df['val_loss'].mean():.4f} ± {results_df['val_loss'].std():.4f}")

results_df.to_csv(DATA_DIR / f"{MODEL_NAME}_cv_results.csv", index=False)
print(f"\nSaved to {DATA_DIR / f'{MODEL_NAME}_cv_results.csv'}")